# Future-Guided Learning — CHB-MIT (Kaggle Edition)

Tái hiện thực nghiệm EEG của bài báo:
> Gunasekaran et al., *A predictive approach to enhance time-series forecasting*, Nature Communications 2025.
> Repo gốc: https://github.com/SkyeGunasekaran/FutureGuidedLearning

---
### Yêu cầu cấu hình Kaggle Notebook:
1. **Accelerator:** Chọn **GPU T4 x2** hoặc **GPU P100** (ở menu bên phải: `Settings` > `Accelerator`).
2. **Internet:** BẮT BUỘC bật **Internet: ON** (`Settings` > `Internet`) để clone repo và tải dữ liệu từ PhysioNet.
3. **Persistence:** Các checkpoint và file kết quả sẽ được ghi vào `/kaggle/working/`.

### Pipeline thực nghiệm:
1. **Stage 1 (Teacher):** Huấn luyện mô hình phát hiện động kinh (*Seizure Detection* — `seizure_detection.py`)
2. **Stage 2 (Baseline):** Huấn luyện mô hình dự báo động kinh không có FGL (*Seizure Prediction* — `seizure_prediction.py`)
3. **Stage 3 (Student + FGL):** Huấn luyện mô hình dự báo động kinh có Teacher chỉ dẫn (`FGL_CHBMIT.py`)
4. **Stage 2b (MViT):** Baseline thứ hai dùng Vision Transformer, để so sánh đủ ba chiều như bài báo gốc

### Phạm vi thực nghiệm: 6 bệnh nhân
`1, 3, 5, 9, 13, 23` — chọn đa dạng theo kết quả bài báo gốc (Table 1): patient 13 đạt AUC cao nhất
(1.00), patient 9 thấp nhất (0.57), patient 23 là trường hợp FGL duy nhất thua CNN-LSTM baseline
(0.80×) — các ca này cho dữ liệu thảo luận phong phú hơn thay vì chỉ chọn ngẫu nhiên. Patient 3, 5
là 2 bệnh nhân đã chạy ở phiên trước, giữ lại để không mất kết quả cũ.

### Giới hạn Kaggle Free & cơ chế resume
GPU free tier giới hạn **~12 giờ/phiên** và **~30 giờ GPU/tuần**. Với ước tính 30 phút–2 giờ/bệnh
nhân cho cả 3 giai đoạn, 6 bệnh nhân có thể chạm trần 1 phiên. Mỗi cell huấn luyện bên dưới **tự
kiểm tra checkpoint/marker đã có trước khi chạy** — nếu notebook bị ngắt giữa chừng (hết giờ
phiên), chỉ cần mở lại và **Run All**: các bệnh nhân đã hoàn thành sẽ tự được bỏ qua, không train
lại từ đầu, giúp tận dụng nhiều phiên free liên tiếp trong tuần.

### Dọn dẹp disk tự động & tự phục hồi cache hỏng
Cache STFT (`cachedir-teacher/`, `cachedir-student/`) của mỗi bệnh nhân có thể nặng vài GB do
oversampling lớp ictal. Notebook tự xoá cache của một bệnh nhân ngay sau khi giai đoạn cần nó
hoàn tất (giữ lại checkpoint + kết quả), tránh tràn disk khi chạy nhiều bệnh nhân liên tiếp. Nếu
từng gặp lỗi `No space left on device` ở phiên trước, chạy cell **Dọn dẹp khẩn cấp** ngay sau
mục 3 trước khi tiếp tục.

Khi disk hết giữa lúc đang **ghi** cache, file `.hickle` còn sót lại bị hỏng (không đọc lại được)
— lần chạy sau sẽ crash nếu không xử lý. Mỗi lệnh huấn luyện bên dưới giờ tự phát hiện lỗi, xoá
cache của đúng bệnh nhân đó, và thử lại đúng 1 lần trước khi báo lỗi thật.


## 1. Clone repository về Kaggle Working Directory

In [ ]:
import os
import shutil

PROJECT_DIR = '/kaggle/working/FutureGuidedLearning'
REPO_URL = 'https://github.com/SkyeGunasekaran/FutureGuidedLearning.git'

if not os.path.isdir(PROJECT_DIR):
    print('Đang clone repo từ GitHub...')
    !git clone "$REPO_URL" "$PROJECT_DIR"
else:
    print('Repo đã tồn tại trong /kaggle/working, bỏ qua clone.')

%cd {PROJECT_DIR}/CHBMIT
!pwd


## 2. Cài đặt các thư viện cần thiết

Kaggle đã cài sẵn PyTorch và CUDA tương thích. Cài thêm các thư viện xử lý tín hiệu EEG (`mne`, `hickle`, `stft`, `scikit-learn`).

In [ ]:
!pip install -q mne==1.5.1 hickle stft tqdm scikit-learn

import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name:', torch.cuda.get_device_name(0))
    print('Device Count:', torch.cuda.device_count())


In [ ]:
%pip install stft hickle mne==1.5.1


## 3. Liên kết dữ liệu CHB-MIT từ Kaggle Dataset

Dữ liệu CHB-MIT từ PhysioNet là mở hoàn toàn. Dataset Kaggle `abhishekinnvonix/...` đã chứa sẵn
toàn bộ 23 bệnh nhân (`chb01`–`chb23`), nên không cần tải thêm — cell dưới chỉ symlink toàn bộ
thư mục `chb*` có sẵn vào `Dataset/` của code, không giới hạn theo một bệnh nhân cụ thể nào.

In [ ]:
# 1. Di chuyển vào thư mục code CHBMIT
%cd /kaggle/working/FutureGuidedLearning/CHBMIT

# 2. Đảm bảo dữ liệu đã liên kết vào Dataset/ của CHBMIT
import os, shutil
DATA_ROOT = "/kaggle/input/datasets/abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0"

os.makedirs("Dataset", exist_ok=True)
for item in os.listdir(DATA_ROOT):
    if item.startswith("chb"):
        src = os.path.join(DATA_ROOT, item)
        dst = os.path.join("Dataset", item)
        if os.path.islink(dst) or os.path.exists(dst):
            os.unlink(dst) if os.path.islink(dst) else shutil.rmtree(dst)
        os.symlink(src, dst)

# 3. Kiểm tra xem module exp và dữ liệu đã sẵn sàng chưa
!ls -d exp Dataset/chb01


In [ ]:
# Danh sách bệnh nhân dùng chung cho cả 3 giai đoạn bên dưới — đổi ở đây là áp dụng cho toàn bộ
# pipeline, không cần sửa rải rác từng cell.
import os

PATIENTS = ['1', '3', '5', '9', '13', '23']

missing = [p for p in PATIENTS if not os.path.isdir(f'Dataset/chb{int(p):02d}')]
if missing:
    print(f'Thiếu dữ liệu cho bệnh nhân: {missing} — kiểm tra lại DATA_ROOT ở cell trên trước khi tiếp tục.')
else:
    print(f'Dữ liệu đầy đủ cho {len(PATIENTS)} bệnh nhân: {PATIENTS}')


In [ ]:
# Cache STFT của mỗi bệnh nhân (cachedir-teacher/, cachedir-student/) có thể nặng vài GB do
# oversampling lớp ictal, và KHÔNG tự dọn qua các lần chạy — đây là nguyên nhân chính gây tràn
# disk trên Kaggle free khi mở rộng nhiều bệnh nhân. Hàm dưới đây xoá cache của một bệnh nhân
# sau khi giai đoạn cần nó đã xong, để giải phóng chỗ cho bệnh nhân tiếp theo.

import subprocess


def cleanup_patient_cache(cachedir, patient):
    freed = 0
    for kind in ("ictal", "interictal"):
        fp = os.path.join(cachedir, f"{kind}_{patient}.hickle")
        if os.path.exists(fp):
            freed += os.path.getsize(fp)
            os.remove(fp)
    if freed:
        print(f"Đã xoá cache {cachedir} cho patient {patient} ({freed / 1e9:.2f} GB).")


def show_disk_usage(label=""):
    total, used, free = shutil.disk_usage("/kaggle/working")
    print(f"Disk {label}: dùng {used/1e9:.1f} GB / tổng {total/1e9:.1f} GB (còn trống {free/1e9:.1f} GB)")


def run_stage_with_retry(cmd, cachedirs, patient, label):
    """Chạy 1 lệnh training; nếu lỗi, coi là khả năng cache .hickle bị ghi dở dang/hỏng
    (thường gặp khi phiên trước hết disk giữa lúc đang ghi) — xoá cache của đúng bệnh nhân
    đó rồi thử lại đúng 1 lần. Nếu lỗi lần 2 thì mới coi là lỗi thật, không retry thêm."""
    env = {**os.environ, "PYTHONPATH": "."}
    ret = subprocess.run(cmd, env=env)
    if ret.returncode != 0:
        print(f"[{label}] Patient {patient}: lỗi lần 1 (exit {ret.returncode}) — "
              f"có thể do cache hỏng từ phiên trước, xoá cache và thử lại 1 lần...")
        for cachedir in cachedirs:
            cleanup_patient_cache(cachedir, patient)
        ret = subprocess.run(cmd, env=env)
        if ret.returncode != 0:
            print(f"[{label}] Patient {patient}: vẫn lỗi sau khi thử lại (exit {ret.returncode}) — "
                  f"xem log phía trên, đây là lỗi thật chứ không phải cache hỏng.")
    return ret


### Dọn dẹp khẩn cấp (chạy 1 lần nếu gặp lỗi `No space left on device`)

Xoá cache STFT của những bệnh nhân **đã hoàn tất** giai đoạn tương ứng — an toàn vì checkpoint/
kết quả đã được lưu, không cần cache gốc nữa trừ khi muốn train lại từ đầu bệnh nhân đó.

**Lưu ý:** nếu một bệnh nhân bị ngắt đúng lúc đang *ghi* cache (như patient 5 ở trên, lỗi ngay
khi tạo `cachedir-student`), file `.hickle` của bệnh nhân đó có thể dở dang/hỏng — không bị cell
dưới xoá (vì marker chưa có), nhưng cũng có thể khiến lần chạy lại lỗi khác (không phải hết disk)
khi cố đọc file hỏng. Nếu gặp tình huống đó, xoá thủ công cache của **đúng bệnh nhân** rồi chạy
lại Stage tương ứng, ví dụ cho patient 5:
```
!rm -f cachedir-student/*_5.hickle cachedir-teacher/*_5.hickle
```

In [ ]:
show_disk_usage("trước khi dọn")

# Teacher: an toàn xoá cache nếu checkpoint đã tồn tại
for p in PATIENTS:
    if os.path.exists(f"pytorch_models/Patient_{p}_detection"):
        cleanup_patient_cache("cachedir-teacher", p)

# Student: an toàn xoá cache nếu CẢ Baseline lẫn FGL của bệnh nhân đó đã xong (cache student dùng chung cho 2 stage)
for p in PATIENTS:
    baseline_done = os.path.exists(f".done_markers/baseline_{p}.done")
    fgl_done = os.path.exists(f".done_markers/fgl_{p}.done")
    if baseline_done and fgl_done:
        cleanup_patient_cache("cachedir-student", p)

show_disk_usage("sau khi dọn")


## 4. Stage 1 — Train Teacher (Seizure Detection)

Mô hình Teacher phát hiện cơn động kinh (ictal vs interictal).
Trọng số mô hình sau khi train sẽ được lưu vào `pytorch_models/Patient_1_detection`.

In [ ]:
import os
import stft

# Sửa numpy.lib.pad thành numpy.pad trong thư viện stft
stft_dir = os.path.dirname(stft.__file__)
utils_path = os.path.join(stft_dir, "utils.py")

with open(utils_path, "r") as f:
    code = f.read()

code = code.replace("numpy.lib.pad", "numpy.pad")

with open(utils_path, "w") as f:
    f.write(code)

print("Đã vá lỗi stft (numpy.lib.pad -> numpy.pad) thành công!")


In [ ]:
%cd /kaggle/working/FutureGuidedLearning/CHBMIT

for p in PATIENTS:
    ckpt = f"pytorch_models/Patient_{p}_detection"
    if os.path.exists(ckpt):
        print(f"[Teacher] Patient {p}: đã có checkpoint ({ckpt}), bỏ qua.")
        continue
    print(f"\n[Teacher] Patient {p}: bắt đầu huấn luyện...")
    cmd = ["python", "exp/seizure_detection.py", "--patient", p, "--epochs", "50", "--patience", "5"]
    ret = run_stage_with_retry(cmd, ["cachedir-teacher"], p, "Teacher")
    if ret.returncode == 0:
        # Cache STFT (cachedir-teacher/ictal_{p}, interictal_{p}) chỉ cần cho Stage 1 — xoá ngay
        # sau khi checkpoint đã lưu để tránh tích luỹ disk qua nhiều bệnh nhân.
        cleanup_patient_cache("cachedir-teacher", p)


## 5. Stage 2 — Train Baseline (Seizure Prediction, KHÔNG có FGL)

Mô hình Baseline dự báo cơn động kinh (preictal vs interictal), không có sự hướng dẫn của Teacher.

In [ ]:
%cd /kaggle/working/FutureGuidedLearning/CHBMIT

os.makedirs(".done_markers", exist_ok=True)
for p in PATIENTS:
    marker = f".done_markers/baseline_{p}.done"
    if os.path.exists(marker):
        print(f"[Baseline] Patient {p}: đã chạy xong trước đó, bỏ qua.")
        continue
    print(f"\n[Baseline] Patient {p}: bắt đầu huấn luyện (3 trials x 25 epochs)...")
    cmd = ["python", "exp/seizure_prediction.py", "--patient", p,
           "--model", "CNN_LSTM", "--trials", "3", "--epochs", "25"]
    ret = run_stage_with_retry(cmd, ["cachedir-student"], p, "Baseline")
    if ret.returncode == 0:
        open(marker, "w").close()


## 6. Stage 3 — Train Student + FGL (Seizure Prediction, CÓ Teacher dẫn đường)

Mô hình Student học dự báo động kinh kết hợp loss distillation từ mô hình Teacher đã lưu ở Stage 1.

In [ ]:
# Sửa torch.load(path) thành torch.load(path, weights_only=False) trong FGL_CHBMIT.py
file_path = "exp/FGL_CHBMIT.py"

with open(file_path, "r") as f:
    code = f.read()

code = code.replace("torch.load(path)", "torch.load(path, weights_only=False)")

with open(file_path, "w") as f:
    f.write(code)

print("Đã vá lỗi PyTorch 2.6 weights_only thành công!")


In [ ]:
%cd /kaggle/working/FutureGuidedLearning/CHBMIT

os.makedirs(".done_markers", exist_ok=True)
for p in PATIENTS:
    marker = f".done_markers/fgl_{p}.done"
    teacher_ckpt = f"pytorch_models/Patient_{p}_detection"
    if os.path.exists(marker):
        print(f"[FGL] Patient {p}: đã chạy xong trước đó, bỏ qua.")
        continue
    if not os.path.exists(teacher_ckpt):
        print(f"[FGL] Patient {p}: thiếu checkpoint Teacher ({teacher_ckpt}) — chạy lại Stage 1 trước. Bỏ qua.")
        continue
    print(f"\n[FGL] Patient {p}: bắt đầu huấn luyện (3 trials x 30 epochs)...")
    cmd = ["python", "exp/FGL_CHBMIT.py", "--patient", p, "--epochs", "30", "--trials", "3",
           "--optimizer", "Adam", "--alpha", "0.5", "--temperature", "4"]
    ret = run_stage_with_retry(cmd, ["cachedir-student"], p, "FGL")
    if ret.returncode == 0:
        open(marker, "w").close()
        # cachedir-student được Stage 2 (Baseline) và Stage 3 (FGL) dùng chung — chỉ xoá ở đây,
        # sau khi cả 2 giai đoạn đã xong, để Stage 3 còn tận dụng cache từ Stage 2 (nhanh hơn).
        if os.path.exists(f".done_markers/baseline_{p}.done"):
            cleanup_patient_cache("cachedir-student", p)


## 6b. So sánh thêm với MViT (bài báo gốc dùng làm baseline thứ hai)

Bài báo gốc so sánh FGL với **cả hai** baseline: CNN-LSTM và MViT (Multi-Channel Vision
Transformer). `models/model.py` đã có sẵn class `MViT` với đúng hyperparameter bài báo dùng
(`patch_size=(5,10), embed_dim=128, num_heads=4, hidden_dim=256, num_layers=4, dropout=0.1`),
và `seizure_prediction.py` đã hỗ trợ `--model MViT` — chỉ chưa từng được gọi. Chạy thêm bước
này để có đủ ba chiều so sánh (MViT / CNN-LSTM / FGL) đúng như Bảng 1 của bài báo gốc.

Chỉ chạy cho các bệnh nhân đã có cache `cachedir-student` sẵn (Baseline CNN-LSTM đã hoàn tất)
để tận dụng lại, không phải build cache từ đầu.

In [ ]:
%cd /kaggle/working/FutureGuidedLearning/CHBMIT

# Chỉ chạy cho bệnh nhân đã có cache cachedir-student sẵn (Baseline CNN-LSTM đã xong)
MVIT_PATIENTS = [p for p in PATIENTS if os.path.exists(f".done_markers/baseline_{p}.done")]
print(f"Bệnh nhân chạy MViT: {MVIT_PATIENTS}")

os.makedirs(".done_markers", exist_ok=True)


def _read_lines(path):
    if os.path.exists(path):
        with open(path) as f:
            return f.readlines()
    return []


for p in MVIT_PATIENTS:
    marker = f".done_markers/mvit_{p}.done"
    if os.path.exists(marker):
        print(f"[MViT] Patient {p}: đã chạy xong trước đó, bỏ qua.")
        continue
    before = _read_lines("Prediction_results.txt")
    print(f"\n[MViT] Patient {p}: bắt đầu huấn luyện (3 trials x 25 epochs)...")
    cmd = ["python", "exp/seizure_prediction.py", "--patient", p,
           "--model", "MViT", "--trials", "3", "--epochs", "25"]
    ret = run_stage_with_retry(cmd, ["cachedir-student"], p, "MViT")
    if ret.returncode == 0:
        # seizure_prediction.py ghi chung vào Prediction_results.txt bất kể model nào, không
        # phân biệt CNN-LSTM/MViT trong nội dung dòng — tách dòng MViT mới sinh ra sang tệp
        # riêng để không lẫn với kết quả Baseline CNN-LSTM đã có.
        after = _read_lines("Prediction_results.txt")
        new_lines = after[len(before):]
        with open("Prediction_MViT_results.txt", "a") as f:
            f.writelines(new_lines)
        with open("Prediction_results.txt", "w") as f:
            f.writelines(before)
        open(marker, "w").close()


## 7. Tổng kết kết quả & Xuất Checkpoints về Kaggle Output

File zip nén weights và file kết quả sẽ hiển thị ngay ở tab **Output** bên phải màn hình Kaggle để bạn tải về máy tính.

In [ ]:
import os
import shutil

# 1. Tạo thư mục tạm để gom file
export_dir = '/kaggle/working/export_results'
if os.path.exists(export_dir):
    shutil.rmtree(export_dir)
os.makedirs(export_dir, exist_ok=True)

# 2. Copy tất cả các file text kết quả (chứa số liệu của toàn bộ PATIENTS)
txt_files = ['Detection_results.txt', 'Prediction_results.txt',
             'Prediction_MViT_results.txt', 'FGL_results.txt']
for txt_file in txt_files:
    if os.path.exists(txt_file):
        shutil.copy(txt_file, export_dir)
        print(f"Đã gom: {txt_file}")

# 3. Đóng gói thành file zip chứa dữ liệu các bệnh nhân
zip_path = f'/kaggle/working/chbmit_{len(PATIENTS)}patients_results'
shutil.make_archive(zip_path, 'zip', export_dir)

print("\n" + "="*50)
print(f"XUẤT THÀNH CÔNG! File: {zip_path}.zip ({len(PATIENTS)} bệnh nhân: {PATIENTS})")
print("Bạn vào tab Data > Output bên phải để tải file zip về máy nhé!")
print("="*50)

# 4. In nhanh kết quả của toàn bộ PATIENTS ra màn hình
print("\nBẢNG TỔNG KẾT KẾT QUẢ ĐÃ LƯU:")
for txt_file in txt_files:
    if os.path.exists(txt_file):
        print(f"\n--- {txt_file} ---")
        with open(txt_file) as f:
            print(f.read().strip())
